In [ ]:
import sys
import numpy as np
import importlib
import dragons_chime_conv_combine
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, ICRS, Galactic
import astropy.units as u
from reproject import reproject_interp

### Read in CHIME and DRAGONS

In [ ]:
# NOTE: using 690 MHz. Need to read in specific file for DRAGONS
hdu_chime   = fits.open('/srv/data/gmims/gmims-lbn/experiments/results_OG/chime_K_U.fits')
hdu_dragons = fits.open('/srv/data/dva/dragons_hoard/no_BLML_500-1000MHz/Mar05_2024_BB_600_to_700_fixed/merged_maps/high/u/After_Ground_Iono_690MHz_high_55.u.fits')

data_chime   = hdu_chime[0].data
data_dragons = hdu_dragons[0].data

print(data_chime.shape)
print(data_dragons.shape)

hdr_chime   = hdu_chime[0].header
hdr_dragons = hdu_dragons[0].header

wcs = WCS(hdr_chime)
print(wcs)

### Get frequency slice from CHIME:

In [ ]:
fpick = 760.9

freq = wcs.all_pix2world(0,0,range(wcs.array_shape[0]),0)[2]/1e6
fidx = abs(fpick-freq).argmin()
print(freq[fidx])

map_chime = data_chime[fidx]
print(map_chime.shape)

plt.imshow(map_chime[820:1400],vmin=-1,vmax=1,cmap='RdBu_r')

### Regrid DRAGONS to CHIME grid and convert to Kelvin

In [ ]:
map_dragons_Jy, footprint = reproject_interp((data_dragons,WCS(hdr_dragons)),wcs.dropaxis(2))

JytoK = np.load('/srv/data/dva/conversion_factor_JytoK_July2024_BB.npy')
idx = abs(JytoK[0]-fpick).argmin()
print(JytoK[0][idx],JytoK[1][idx])

map_dragons = map_dragons_Jy*JytoK[1][idx]

print(map_dragons.shape)

### Make mask to zero-pad

In [ ]:
ypix = np.arange(0,map_chime.shape[0],1)

Rmax1 = 1400
Rmin1 = 1360
b = np.pi/(Rmax1-Rmin1)
r0 = (Rmax1+Rmin1)/2.
mask1 = -0.5*np.sin(b*(ypix - r0))+0.5
mask1[ypix<Rmin1] = 0
mask1[ypix>Rmax1] = 0

Rmax2 = 860
Rmin2 = 820
b = np.pi/(Rmax2-Rmin2)
r0 = (Rmax2+Rmin2)/2.
mask2 = 0.5*np.sin(b*(ypix - r0))+0.5
mask2[ypix<Rmin2] = 0
mask2[(ypix>Rmax2) & (ypix<Rmin1)] = 1
mask2[ypix>=Rmin1] = 0

mask = mask1+mask2

mask2Dy = np.empty_like(map_chime)

for i in range(0,mask2Dy.shape[1]):   
    mask2Dy[:,i] = mask


xpix = np.arange(0,map_chime.shape[1],1)

Rmax1 = 2870
Rmin1 = 2830
b = np.pi/(Rmax1-Rmin1)
r0 = (Rmax1+Rmin1)/2.
mask1 = -0.5*np.sin(b*(xpix - r0))+0.5
mask1[xpix<Rmin1] = 0
mask1[xpix>Rmax1] = 0

Rmax2 = 50
Rmin2 = 10
b = np.pi/(Rmax2-Rmin2)
r0 = (Rmax2+Rmin2)/2.
mask2 = 0.5*np.sin(b*(xpix - r0))+0.5
mask2[xpix<Rmin2] = 0
mask2[(xpix>Rmax2) & (xpix<Rmin1)] = 1
mask2[xpix>=Rmin1] = 0

mask = mask1+mask2

mask2Dx = np.empty_like(map_chime)

for i in range(0,map_chime.shape[0]):   
    mask2Dx[i,:] = mask

fig,ax = plt.subplots(1,2,figsize=(20,8))
ax[0].imshow(mask2Dy[800:1440]*mask2Dx[800:1440],cmap='viridis',origin='lower')
ax[1].imshow(map_chime[800:1440],cmap='RdBu_r',vmin=-1,vmax=1,origin='lower')


### Apply zero-padding mask

In [ ]:
map_chime[np.isnan(map_chime)] = 0.0
map_dragons[np.isnan(map_dragons)] = 0.0

map_chime_masked   = np.zeros([2880,2880])
map_dragons_masked = np.zeros([2880,2880])

map_chime_masked[1120:1120+640]   = map_chime[800:1440]*mask2Dy[800:1440]*mask2Dx[800:1440]
map_dragons_masked[1120:1120+640] = map_dragons[800:1440]*mask2Dy[800:1440]*mask2Dx[800:1440]

#map_chime_masked   = np.zeros([1160,1160])
#map_dragons_masked = np.zeros([1160,1160])

#map_chime_masked[0:580,0:580] = map_chime[820:1400,2300:2880]
#map_chime_masked[580:580*2,0:580] = map_chime[820:1400,2300:2880][::-1]
#map_chime_masked[:,580:580*2] = map_chime_masked[:,0:580][:,::-1]

#map_dragons_masked[0:580,0:580] = map_dragons[820:1400,2300:2880]
#map_dragons_masked[580:580*2,0:580] = map_dragons[820:1400,2300:2880][::-1]
#map_dragons_masked[:,580:580*2] = map_dragons_masked[:,0:580][:,::-1]

fig,ax = plt.subplots(1,2,figsize=(20,8))
ax[0].imshow(map_chime_masked,cmap='RdBu_r',vmin=-0.5,vmax=0.5,origin='lower')
ax[1].imshow(map_dragons_masked,cmap='RdBu_r',vmin=-0.5,vmax=0.5,origin='lower')

### Calculate the FFTs (and SA adjustments for DRAGONS)

In [ ]:
importlib.reload(dragons_chime_conv_combine)

dxy = hdr_chime['CDELT2']
print(dxy)

map_FFT_chime, mask_chime, beam_chime, xuv_chime, yuv_chime, map_deconv_chime = dragons_chime_conv_combine.SA_deconvolve(map_chime_masked, 5.0, dxy, freq[fidx])
map_FFT_dragons, mask_dragons, beam_dragons, xuv_dragons, yuv_dragons, map_deconv_dragons = dragons_chime_conv_combine.SA_deconvolve(map_dragons_masked, 5.0, dxy, freq[fidx])
print(map_FFT_chime.shape)


In [ ]:
fig,ax = plt.subplots(2,1,figsize=(20,10))
ax[0].imshow(abs(map_FFT_chime),cmap='cubehelix',origin='lower',vmin=0,vmax=5000)
ax[1].imshow(abs(map_FFT_dragons),cmap='cubehelix',origin='lower',vmin=0.0,vmax=100)

In [ ]:
fig,ax = plt.subplots(2,1,figsize=(20,10))

ax[0].plot(xuv_chime[1439],np.nanmean(abs(map_FFT_chime)[1439:1441],axis=0),linewidth=2)
ax[0].plot(xuv_dragons[1439],np.nanmean(abs(map_FFT_dragons)[1439:1441],axis=0),linewidth=2)
ax[0].set_xlim(0,200)
ax[0].set_ylim(0,1e4)
ax[0].axvline(x=0.3,color='k')
ax[0].axvline(x=5,color='k')
#ax[0].set_yscale('log')

ax[1].plot(yuv_chime[:,1439],np.nanmean(abs(map_FFT_chime)[:,1439:1441],axis=1),linewidth=2)
ax[1].plot(yuv_dragons[:,1439],np.nanmean(abs(map_FFT_dragons)[:,1439:1441],axis=1),linewidth=2)
ax[1].plot(yuv_dragons[:,1439],np.nanmean(1e5*beam_dragons[:,1439:1441],axis=1),linewidth=2)
ax[1].plot(yuv_dragons[:,1439],np.nanmean(abs(map_FFT_dragons)[:,1439:1441],axis=1)/np.nanmean(beam_dragons[:,1439:1441],axis=1),linewidth=2)
ax[1].set_xlim(0,20)
ax[1].set_ylim(0,1e5)
ax[1].axvline(x=0.3,color='k')
ax[1].axvline(x=5,color='k')
#ax[1].set_yscale('log')

#ax.axvline(x=21,color='k')
#ax.axvline(x=42,color='k')
#ax.axvline(x=63,color='k')
#ax.set_yscale('log')

In [ ]:
print(xuv_dragons[0,:])
print(yuv_dragons[:,0])

In [ ]:
fig,ax = plt.subplots(2,1,figsize=(5,12))

ax[0].imshow(mask_dragons[1440-200:1440+200,1440-200:1440+200],vmin=0,vmax=1,cmap='viridis',origin='lower')
ax[1].imshow(beam_dragons[1440-200:1440+200,1440-200:1440+200],vmin=0,vmax=1,cmap='viridis',origin='lower')

In [ ]:
fig,ax = plt.subplots(3,1,figsize=(20,25))

ax[0].imshow(map_dragons_masked,vmin=-1.5,vmax=1.5,cmap='RdBu_r',origin='lower')
ax[1].imshow(map_deconv_dragons,vmin=-1.5,vmax=1.5,cmap='RdBu_r',origin='lower')
ax[2].imshow(map_dragons_masked-map_deconv_dragons,vmin=-1,vmax=1,cmap='RdBu_r',origin='lower')